# Pipeline de amostragem para dados desbalanceados — Bloco 3: decisão, modelo final e estimativa final

Este bloco recebe o `contexto` depois do bloco 2 e faz as três últimas etapas do roteiro:

1. **Decisão da técnica no OOT de seleção.** O baseline e cada técnica da lista curta são treinados com as mesmas configurações de hiperparâmetros e as mesmas sementes. A diferença para o baseline é medida com bootstrap pareado por chave.
2. **Modelo final.** O treino da técnica vencedora vai para a sua função do AutoML (Optuna no teste). Sem ela, é usado um Optuna interno com o LightGBM em pandas.
3. **Estimativa final no OOT final**, aberto uma única vez: métricas com intervalo, por safra, grade operacional, matrizes de confusão, calibração corrigida pelo β da técnica e PSI.

**Pré-requisitos, na mesma sessão:** as células dos blocos 1 e 2 executadas e o `contexto` depois de `executar_triagem(contexto)`.

| Célula | Conteúdo |
|---|---|
| 1 | Parâmetros do bloco 3 |
| 2 | Bases: OOT de seleção e OOT final com identificador de linha |
| 3 | Decisão da técnica no OOT de seleção |
| 4 | Modelo final: AutoML do usuário ou Optuna interno |
| 5 | Estimativa final no OOT final |
| 6 | Orquestração e registro |
| 7 | Exemplo de uso |

Convenções dos blocos anteriores continuam valendo: só funções, sem classes; imune a `from pyspark.sql.functions import *`.

## Célula 1 — Parâmetros do bloco 3

Os parâmetros dos blocos 1 e 2 continuam valendo (`n_configuracoes_comuns` = 5, `sementes_decisao` = 2, `n_trials_optuna` = 50, `taxas_alerta_relatorio`, `n_bootstrap`).

| Parâmetro | Padrão | Função |
|---|---|---|
| `espaco_hiperparametros` | ver o código | Faixas de onde são sorteadas as configurações comuns da decisão e os *trials* do Optuna interno |
| `num_iteracoes_decisao` | 500 | Máximo de iterações por treino, com parada antecipada no teste |
| `minimo_configuracoes_com_ganho` | 3 | De 5 configurações, em quantas a técnica precisa superar o baseline (média das sementes) |
| `minimo_fracao_safras_com_ganho` | 0,5 | Fração mínima de safras do OOT de seleção em que a técnica supera o baseline |
| `n_faixas_calibracao` | 10 | Faixas usadas no erro de calibração (ECE) |
| `n_faixas_psi` | 10 | Faixas do PSI, definidas pelos decis do score no teste |
| `forcar_reabertura_oot_final` | `False` | Com `False`, o bloco recusa medir o OOT final uma segunda vez |

`fracaoMinDataInLeaf` é o mínimo de linhas por folha expresso como fração do treino. Assim, a mesma configuração tem o mesmo efeito num treino subamostrado de 16 mil linhas e num treino completo de 800 mil linhas: 0,001 vira 16 linhas no primeiro e 800 no segundo (com mínimo de 5).

In [ ]:
import math

PARAMETROS_BLOCO3 = {
    "espaco_hiperparametros": {
        "numLeaves": [15, 31, 63, 127],          # sorteio entre os valores
        "learningRate": [0.03, 0.15],            # log-uniforme
        "featureFraction": [0.6, 1.0],           # uniforme
        "baggingFraction": [0.6, 1.0],           # uniforme
        "lambdaL2": [0.1, 10.0],                 # log-uniforme
        "fracaoMinDataInLeaf": [0.0001, 0.002],  # log-uniforme
    },
    "num_iteracoes_decisao": 500,
    "minimo_configuracoes_com_ganho": 3,
    "minimo_fracao_safras_com_ganho": 0.5,
    "n_faixas_calibracao": 10,
    "n_faixas_psi": 10,
    "forcar_reabertura_oot_final": False,
}


def montar_parametros_bloco3(contexto, **sobrescritos):
    if "parametros_bloco2" not in contexto:
        raise ValueError("Execute o bloco 2 (executar_triagem) antes do bloco 3.")
    conhecidos = set(PARAMETROS_PADRAO) | set(PARAMETROS_BLOCO2) | set(PARAMETROS_BLOCO3)
    desconhecidos = set(sobrescritos) - conhecidos
    if desconhecidos:
        raise ValueError(f"Parâmetros desconhecidos: {sorted(desconhecidos)}")
    parametros = {k: (dict(v) if isinstance(v, dict) else v) for k, v in contexto["parametros_bloco2"].items()}
    for k, v in PARAMETROS_BLOCO3.items():
        parametros.setdefault(k, dict(v) if isinstance(v, dict) else v)
    for k, v in sobrescritos.items():
        if isinstance(v, dict) and isinstance(parametros.get(k), dict):
            parametros[k].update(v)
        else:
            parametros[k] = v
    return parametros


def sortear_configuracao(espaco, rng):
    def log_uniforme(faixa):
        return float(math.exp(rng.uniform(math.log(faixa[0]), math.log(faixa[1]))))
    return {
        "numLeaves": int(rng.choice(espaco["numLeaves"])),
        "learningRate": log_uniforme(espaco["learningRate"]),
        "featureFraction": float(rng.uniform(*espaco["featureFraction"])),
        "baggingFraction": float(rng.uniform(*espaco["baggingFraction"])),
        "lambdaL2": log_uniforme(espaco["lambdaL2"]),
        "fracaoMinDataInLeaf": log_uniforme(espaco["fracaoMinDataInLeaf"]),
    }


def parametros_com_configuracao(parametros, configuracao, n_linhas, num_iteracoes):
    """Cópia dos parâmetros com o proxy ajustado à configuração e ao tamanho do treino."""
    novo = dict(parametros)
    proxy = dict(parametros["proxy"])
    proxy.update({k: v for k, v in configuracao.items() if k != "fracaoMinDataInLeaf"})
    proxy["minDataInLeaf"] = int(builtins.max(5, builtins.round(configuracao["fracaoMinDataInLeaf"] * n_linhas)))
    proxy["baggingFreq"] = 1 if proxy["baggingFraction"] < 1 else 0
    proxy["numIterations"] = int(num_iteracoes)
    novo["proxy"] = proxy
    return novo

## Célula 2 — Bases do bloco 3

Reaproveita as tabelas do bloco 2 (`base_treino`, `base_teste`, `amostra_avaliacao_treino` e as análises) e grava o OOT de seleção e o OOT final com a coluna `_id_linha`:

| Tabela | Conteúdo |
|---|---|
| `base_oot_selecao` | OOT de seleção do bloco 1 com `_id_linha` |
| `base_oot_final` | OOT final do bloco 1 com `_id_linha` |

Se alguma técnica da lista curta usa vizinhança, a padronização e os vizinhos dos positivos são recalculados com as mesmas features de distância do bloco 2.

In [ ]:
def _tabela_com_id(contexto, parte, nome, parametros):
    col = contexto["colunas"]
    destino, prefixo = contexto["destino_delta"], parametros["prefixo_tabelas"]
    df_parte = contexto["partes"][parte]
    esperado = df_parte.columns + ["_id_linha"]
    contagem = calcular_prevalencia(df_parte, col["alvo"])
    if parametros["retomar"]:
        try:
            existente = spark.table(nome_tabela(destino, prefixo, nome))
            if existente.columns == esperado and calcular_prevalencia(existente, col["alvo"]) == contagem:
                log_pipeline(f"Reaproveitando a tabela {nome}.")
                return existente
        except Exception:
            pass
    return salvar_tabela(df_parte.withColumn("_id_linha", F.monotonically_increasing_id()),
                         destino, prefixo, nome, contexto["registro"])


def _info_avaliacao(df, colunas):
    info = (df.select("_id_linha", coluna(colunas["chave"]), coluna(colunas["safra"]), coluna(colunas["alvo"]))
              .toPandas().sort_values("_id_linha").reset_index(drop=True))
    return {
        "ids": info["_id_linha"].to_numpy(),
        "y": info[colunas["alvo"]].to_numpy(dtype=np.float64),
        "safra": info[colunas["safra"]].astype(str).to_numpy(),
        "codigos_chave": pd.factorize(info[colunas["chave"]])[0],
        "chave": info[colunas["chave"]].to_numpy(),
        "safra_original": info[colunas["safra"]].to_numpy(),
    }


def preparar_bases_bloco3(contexto, parametros):
    base = preparar_tabelas_triagem(contexto, parametros, contexto["metrica"])
    if not base["reaproveitar"]:
        contexto["registro"]["avisos"].append(
            "Bloco 3: as tabelas do bloco 2 foram recriadas; os identificadores de linha mudaram desde a triagem.")
    base["oot_selecao"] = _tabela_com_id(contexto, "oot_selecao", "base_oot_selecao", parametros)
    base["oot_final"] = _tabela_com_id(contexto, "oot_final", "base_oot_final", parametros)
    base["info_oot_selecao"] = _info_avaliacao(base["oot_selecao"], contexto["colunas"])
    base["info_teste"] = _info_avaliacao(base["teste"], contexto["colunas"])

    precisa_vizinhanca = any(TECNICAS[x["tecnica"]]["distancia"] or TECNICAS[x["tecnica"]]["driver"]
                             for x in contexto["lista_curta"])
    if precisa_vizinhanca:
        preparar_vizinhanca(base, contexto["features_distancia"])
    return base


def _scores_parte(base, proxy, parte):
    """Scores alinhados aos ids ordenados da parte ('teste', 'oot_selecao' ou 'oot_final')."""
    p = base["parametros"]
    cache = base["cache_pandas"]
    chave_cache = f"matriz_{parte}"
    ids = base[f"info_{parte}"]["ids"]
    if proxy.get("motor") == "pandas":
        if chave_cache not in cache:
            cache[chave_cache] = matriz_pandas(base[parte], base["features"], manter=["_id_linha"])
        return prever_pandas(proxy, cache[chave_cache])
    sc = pontuar(proxy, base[parte], ["_id_linha"], p).toPandas()
    return pd.Series(sc["score"].to_numpy(), index=sc["_id_linha"].to_numpy()).reindex(ids).to_numpy()

## Célula 3 — Decisão da técnica no OOT de seleção

**Grade da decisão:** técnicas da lista curta (o baseline sempre incluído) × `n_configuracoes_comuns` configurações × `sementes_decisao` sementes. Com 3 técnicas além do baseline: 4 × 5 × 2 = 40 treinos.

- As configurações são sorteadas uma vez, com semente fixa, e são **as mesmas para todas as técnicas**. Assim, a diferença entre técnicas não vem de hiperparâmetros diferentes.
- A semente muda o sorteio da técnica (nas técnicas com sorteio) e o modelo (amostragem de linhas e colunas do LightGBM).
- Cada treino usa parada antecipada no **teste** e é medido no **OOT de seleção**, pela métrica escolhida no bloco 1.

**Comparação com o baseline**, para cada técnica:

| Medida | Cálculo |
|---|---|
| `diferenca_baseline` | Média da métrica nos 10 treinos da técnica − média nos 10 treinos do baseline |
| `ic_inferior`, `ic_superior` | Intervalo de 95% da diferença por bootstrap pareado por chave |
| `configuracoes_com_ganho` | Em quantas das 5 configurações a técnica supera o baseline, na média das sementes |
| `fracao_safras_com_ganho` | Fração das safras do OOT de seleção em que a técnica supera o baseline |

**Regra de decisão**, definida antes de rodar: vence a técnica de maior `diferenca_baseline` entre as que têm `ic_inferior` > 0, pelo menos `minimo_configuracoes_com_ganho` configurações com ganho e pelo menos `minimo_fracao_safras_com_ganho` das safras com ganho. Se nenhuma atender às três condições, vence o baseline: não amostrar é a opção mais simples, e a técnica precisa provar que é melhor.

Os treinos são gravados em `decisao_resultados` e os scores em `decisao_scores`. Com `retomar=True`, os treinos concluídos são pulados.

In [ ]:
def esquema_decisao(taxas):
    campos = [("config_id", T.StringType()), ("tecnica", T.StringType()), ("nivel", T.DoubleType()),
              ("configuracao", T.LongType()), ("semente", T.LongType()), ("status", T.StringType()),
              ("motivo", T.StringType()), ("metrica_decisao", T.DoubleType()), ("ks", T.DoubleType()),
              ("auc_roc", T.DoubleType()), ("gini", T.DoubleType()), ("pr_auc", T.DoubleType())]
    campos += [(c, T.DoubleType()) for c in colunas_operacionais(taxas)]
    campos += [("por_safra", T.StringType()), ("hiperparametros", T.StringType()), ("info", T.StringType()),
               ("segundos", T.DoubleType()), ("momento", T.StringType())]
    return T.StructType([T.StructField(n, t, True) for n, t in campos])


def _id_decisao(tecnica, nivel, configuracao, semente):
    return f"{tecnica}|{'-' if nivel is None else f'{nivel:g}'}|c{configuracao}|{semente}"


def executar_grade_decisao(contexto, base, parametros):
    metrica = contexto["metrica"]
    nome = NOME_METRICA[metrica]
    taxas = parametros["taxas_alerta_relatorio"]
    esquema = esquema_decisao(taxas)
    tabelas = {"resultados": nome_tabela(base["destino"], base["prefixo"], "decisao_resultados"),
               "scores": nome_tabela(base["destino"], base["prefixo"], "decisao_scores")}
    rng = np.random.default_rng(parametros["semente"] + 31)
    configuracoes = [sortear_configuracao(parametros["espaco_hiperparametros"], rng)
                     for _ in range(parametros["n_configuracoes_comuns"])]
    sementes = [parametros["semente"] + 104729 * (i + 1) for i in range(parametros["sementes_decisao"])]

    feitos = set()
    if parametros["retomar"] and base["reaproveitar"]:
        try:
            anteriores = spark.table(tabelas["resultados"])
            if [(c.name, c.dataType) for c in anteriores.schema.fields] == [(c.name, c.dataType) for c in esquema.fields]:
                feitos = {r["config_id"] for r in anteriores.filter(F.col("status") == "ok").select("config_id").collect()}
        except Exception:
            feitos = set()
    estado = {"resultados": not feitos, "scores": not feitos}

    total = len(contexto["lista_curta"]) * len(configuracoes) * len(sementes)
    log_pipeline(f"Decisão: {total} treinos ({len(contexto['lista_curta'])} técnicas × {len(configuracoes)} "
                 f"configurações × {len(sementes)} sementes); {len(feitos)} já concluídos.")
    inicio, n_feitos = time.time(), 0
    info_oot = base["info_oot_selecao"]
    for item in contexto["lista_curta"]:
        tecnica, nivel = item["tecnica"], item["nivel"]
        for semente in sementes:
            ids = [_id_decisao(tecnica, nivel, c, semente) for c in range(len(configuracoes))]
            if all(i in feitos for i in ids):
                continue
            semente_tecnica = semente if TECNICAS[tecnica]["sorteio"] else parametros["semente"]
            resultado = aplicar_tecnica(base, tecnica, nivel, semente_tecnica, metrica)
            df_tecnica = resultado["df"].persist()
            n_linhas = int(builtins.round(resultado["n_pos"])) + resultado["n_neg"]
            try:
                for c, configuracao in enumerate(configuracoes):
                    cid = ids[c]
                    if cid in feitos:
                        continue
                    t0 = time.time()
                    valores = {"config_id": cid, "tecnica": tecnica, "nivel": nivel, "configuracao": c,
                               "semente": semente, "momento": datetime.now().isoformat(timespec="seconds"),
                               "hiperparametros": json.dumps(configuracao)}
                    try:
                        p_cfg = parametros_com_configuracao(parametros, configuracao, n_linhas,
                                                            parametros["num_iteracoes_decisao"])
                        proxy = treinar_proxy(df_tecnica, base["teste"], base["features"], base["alvo"], metrica,
                                              semente, p_cfg, col_peso=resultado["col_peso"], cache=base["cache_pandas"])
                        s = _scores_parte(base, proxy, "oot_selecao")
                        m = calcular_metricas(info_oot["y"], s, None, taxas)
                        valores.update(m)
                        valores.update(
                            status="ok", metrica_decisao=m[nome],
                            por_safra=json.dumps(metricas_por_safra(info_oot["y"], s, info_oot["safra"], metrica)),
                            info=json.dumps({"motor": proxy["motor"], "iteracoes": proxy.get("iteracoes"),
                                             "min_data_in_leaf": p_cfg["proxy"]["minDataInLeaf"]}))
                        scores = spark.createDataFrame(pd.DataFrame({
                            "config_id": cid, "_id_linha": info_oot["ids"].astype(np.int64), "score": s}))
                        _gravar(scores, tabelas["scores"], estado, "scores")
                    except Exception as erro:
                        valores.update(status="erro", motivo=f"{type(erro).__name__}: {erro}"[:2000])
                    valores["segundos"] = time.time() - t0
                    _gravar(spark.createDataFrame([_linha_resultado(esquema, valores)], esquema),
                            tabelas["resultados"], estado, "resultados")
                    n_feitos += 1
                    restante = (time.time() - inicio) / n_feitos * builtins.max(total - len(feitos) - n_feitos, 0)
                    detalhe = (f"{metrica} = {valores['metrica_decisao']:.4f}" if valores["status"] == "ok"
                               else f"erro: {valores['motivo'][:200]}")
                    log_pipeline(f"[decisão] {cid}: {detalhe} | restante estimado: {restante / 60:.0f} min")
            finally:
                df_tecnica.unpersist()
                liberar_resultado(resultado)
    return configuracoes, tabelas


def resumir_decisao(contexto, base, parametros, tabelas):
    metrica = contexto["metrica"]
    info_oot = base["info_oot_selecao"]
    resultados = spark.table(tabelas["resultados"]).toPandas()
    ok = resultados[resultados["status"] == "ok"].copy()
    erros = resultados[resultados["status"] == "erro"]
    if len(erros):
        contexto["registro"]["avisos"].append(f"Bloco 3: {len(erros)} treinos da decisão terminaram com erro.")
    ok["nivel_txt"] = ok["nivel"].map(lambda v: "-" if pd.isna(v) else f"{v:g}")
    base_ok = ok[ok["tecnica"] == "baseline"]
    if base_ok.empty:
        raise RuntimeError("Nenhum treino do baseline foi concluído na decisão.")

    lista_ids = spark.createDataFrame(pd.DataFrame({"config_id": ok["config_id"].tolist()}))
    scores = (spark.table(tabelas["scores"]).join(lista_ids, on="config_id", how="left_semi").toPandas()
              .drop_duplicates(["config_id", "_id_linha"], keep="last"))
    por_config = {cid: pd.Series(g["score"].to_numpy(), index=g["_id_linha"].to_numpy()).reindex(info_oot["ids"]).to_numpy()
                  for cid, g in scores.groupby("config_id")}

    def media_por_cfg(g):
        return g.groupby("configuracao")["metrica_decisao"].mean()

    def safras_medias(g):
        return pd.DataFrame([json.loads(x) for x in g["por_safra"]]).mean()

    base_cfg, base_safras = media_por_cfg(base_ok), safras_medias(base_ok)
    valor_base = float(base_ok["metrica_decisao"].mean())
    linhas = []
    for (tecnica, nivel_txt), g in ok.groupby(["tecnica", "nivel_txt"], sort=False):
        linha = {"tecnica": tecnica, "nivel": g["nivel"].iloc[0], "treinos": len(g),
                 "metrica_media": float(g["metrica_decisao"].mean()),
                 "metrica_desvio": float(g["metrica_decisao"].std(ddof=1)) if len(g) > 1 else np.nan,
                 "diferenca_baseline": float(g["metrica_decisao"].mean()) - valor_base,
                 "ks": g["ks"].mean(), "auc_roc": g["auc_roc"].mean(), "pr_auc": g["pr_auc"].mean()}
        for c in colunas_operacionais(parametros["taxas_alerta_relatorio"]):
            linha[c] = g[c].mean()
        if tecnica == "baseline":
            linha.update(configuracoes_com_ganho=np.nan, fracao_safras_com_ganho=np.nan,
                         ic_inferior=np.nan, ic_superior=np.nan, prob_ganho=np.nan)
        else:
            cfg = media_por_cfg(g)
            comuns = cfg.index.intersection(base_cfg.index)
            linha["configuracoes_com_ganho"] = int((cfg[comuns] > base_cfg[comuns]).sum())
            saf = safras_medias(g)
            comparaveis = [s for s in saf.index if s in base_safras.index
                           and np.isfinite(saf[s]) and np.isfinite(base_safras[s])]
            linha["fracao_safras_com_ganho"] = (float(np.mean([saf[s] > base_safras[s] for s in comparaveis]))
                                               if comparaveis else np.nan)
            r = bootstrap_pareado(info_oot["y"], info_oot["codigos_chave"],
                                  [por_config[i] for i in g["config_id"]],
                                  [por_config[i] for i in base_ok["config_id"]],
                                  metrica, parametros["n_bootstrap"], parametros["semente"])
            linha.update(ic_inferior=r["ic_inferior"], ic_superior=r["ic_superior"], prob_ganho=r["prob_ganho"])
        linhas.append(linha)
    resumo = pd.DataFrame(linhas)

    def aprovada(r):
        motivos = []
        if not (r["ic_inferior"] > 0):
            motivos.append("o intervalo de 95% da diferença inclui zero ou é negativo")
        if not (r["configuracoes_com_ganho"] >= parametros["minimo_configuracoes_com_ganho"]):
            motivos.append(f"ganho em só {int(r['configuracoes_com_ganho'])} de {parametros['n_configuracoes_comuns']} configurações")
        if not (r["fracao_safras_com_ganho"] >= parametros["minimo_fracao_safras_com_ganho"]):
            motivos.append(f"ganho em só {r['fracao_safras_com_ganho']:.0%} das safras")
        return motivos

    resumo["motivos_reprovacao"] = [None if r["tecnica"] == "baseline" else "; ".join(aprovada(r)) or None
                                    for _, r in resumo.iterrows()]
    resumo["aprovada"] = [r["tecnica"] != "baseline" and r["motivos_reprovacao"] is None for _, r in resumo.iterrows()]
    resumo = resumo.sort_values("diferenca_baseline", ascending=False).reset_index(drop=True)
    aprovadas = resumo[resumo["aprovada"]]
    escolhida = aprovadas.iloc[0] if len(aprovadas) else resumo[resumo["tecnica"] == "baseline"].iloc[0]
    nivel = None if pd.isna(escolhida["nivel"]) else float(escolhida["nivel"])
    item = next(x for x in contexto["lista_curta"]
                if x["tecnica"] == escolhida["tecnica"] and (x["nivel"] == nivel or (x["nivel"] is None and nivel is None)))
    vencedora = dict(item)
    vencedora["diferenca_oot_selecao"] = float(escolhida["diferenca_baseline"])
    vencedora["motivo"] = (f"Maior diferença entre as técnicas aprovadas ({len(aprovadas)} aprovada(s))."
                           if len(aprovadas) else
                           "Nenhuma técnica atendeu às três condições da regra; vence o baseline (sem amostragem).")
    log_pipeline(f"Técnica vencedora: {vencedora['tecnica']}"
                 + ("" if vencedora["nivel"] is None else f" (nível {vencedora['nivel']:g})")
                 + f". {vencedora['motivo']}")
    return resultados, resumo, vencedora

## Célula 4 — Modelo final

**Com a sua função do AutoML** (`funcao_automl`), ela é chamada assim:

```
modelo = funcao_automl(treino=..., teste=..., oot=..., features=..., metrica=..., n_trials=..., **kwargs_automl)
```

| Argumento | Conteúdo |
|---|---|
| `treino` | DataFrame Spark da técnica vencedora (`treino_<tecnica>`): chave, safra, alvo e features |
| `teste` | Teste do bloco 1, com a distribuição real |
| `oot` | **OOT de seleção** (o OOT final nunca é entregue ao AutoML) |
| `features` | Lista das features selecionadas no bloco 1 |
| `metrica` | A string escolhida no bloco 1 |
| `n_trials` | `n_trials_optuna` |

Se o AutoML tiver outros argumentos (por exemplo, o nome do alvo), passe em `kwargs_automl`. Se os nomes dos argumentos forem diferentes, basta uma função pequena que traduza.

O espaço de busca do Optuna do AutoML **não deve conter `scale_pos_weight` nem `is_unbalance`**: o desbalanceamento já foi tratado pela técnica, e o β da correção de calibração depende disso.

Para pontuar, o bloco precisa de uma `funcao_score(modelo, df)` que devolva o mesmo `df` com uma coluna `score` (probabilidade da classe positiva) acrescentada. Sem ela, o bloco tenta `modelo.transform(df)` e lê a coluna `probability`.

**Sem `funcao_automl`**, é usado um Optuna interno: amostrador TPE (*Tree-structured Parzen Estimator*) com semente fixa, `n_trials_optuna` *trials* no mesmo espaço de hiperparâmetros da decisão, LightGBM em pandas com parada antecipada no teste e a métrica do bloco 1 como objetivo. O melhor *trial* é retreinado e devolvido no mesmo formato do *proxy*.

In [ ]:
def treinar_modelo_final_interno(treino, teste, features, col_alvo, metrica, parametros):
    import optuna

    nome = NOME_METRICA[metrica]
    linhas_treino = treino.count()
    memoria = _megabytes(linhas_treino, features) + _megabytes(teste.count(), features)
    if memoria > parametros["limite_memoria_proxy_mb"]:
        raise MemoryError(f"Treino e teste ocupariam cerca de {memoria:,.0f} MB em pandas, acima de "
                          f"limite_memoria_proxy_mb. Use funcao_automl (seu AutoML em Spark).")
    treino_pd = _matriz_spark(treino, features, col_alvo=col_alvo).toPandas()
    teste_pd = _matriz_spark(teste, features, col_alvo=col_alvo).toPandas()
    y_teste = teste_pd["_rotulo"].to_numpy(dtype=np.float64)
    espaco = parametros["espaco_hiperparametros"]

    def configuracao_do_trial(trial):
        return {
            "numLeaves": trial.suggest_categorical("numLeaves", list(espaco["numLeaves"])),
            "learningRate": trial.suggest_float("learningRate", *espaco["learningRate"], log=True),
            "featureFraction": trial.suggest_float("featureFraction", *espaco["featureFraction"]),
            "baggingFraction": trial.suggest_float("baggingFraction", *espaco["baggingFraction"]),
            "lambdaL2": trial.suggest_float("lambdaL2", *espaco["lambdaL2"], log=True),
            "fracaoMinDataInLeaf": trial.suggest_float("fracaoMinDataInLeaf", *espaco["fracaoMinDataInLeaf"], log=True),
        }

    def treinar(configuracao):
        p_cfg = parametros_com_configuracao(parametros, configuracao, linhas_treino, parametros["num_iteracoes_decisao"])
        return treinar_proxy_pandas(treino_pd, teste_pd, features, metrica, parametros["semente"], p_cfg)

    def objetivo(trial):
        modelo = treinar(configuracao_do_trial(trial))
        valor = calcular_metricas(y_teste, prever_pandas(modelo, teste_pd))[nome]
        trial.set_user_attr("iteracoes", modelo["iteracoes"])
        return valor

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    estudo = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=parametros["semente"]))
    estudo.optimize(objetivo, n_trials=int(parametros["n_trials_optuna"]))
    melhor = dict(estudo.best_params)
    modelo = treinar(melhor)
    modelo["hiperparametros"] = melhor
    modelo["valor_teste_optuna"] = float(estudo.best_value)
    modelo["historico_optuna"] = estudo.trials_dataframe()
    log_pipeline(f"Optuna interno: {len(estudo.trials)} trials; melhor {metrica} no teste = {estudo.best_value:.4f}.")
    return modelo


def pontuar_modelo_final(modelo, df, manter, parametros, funcao_score=None):
    """DataFrame Spark com as colunas de `manter` e `score`."""
    if funcao_score is not None:
        return funcao_score(modelo, df).select(*[coluna(c) for c in manter], F.col("score").cast("double"))
    if isinstance(modelo, dict) and "motor" in modelo:
        return pontuar(modelo, df, manter, parametros)
    if hasattr(modelo, "transform"):
        saida = modelo.transform(df)
        return saida.select(*[coluna(c) for c in manter], vector_to_array("probability")[1].alias("score"))
    raise ValueError("Não sei pontuar este modelo: informe funcao_score(modelo, df).")


def scores_alinhados(modelo, df, ids, parametros, funcao_score=None):
    if isinstance(modelo, dict) and modelo.get("motor") == "pandas" and funcao_score is None:
        pdf = matriz_pandas(df, modelo["features"], manter=["_id_linha"])
        return pd.Series(prever_pandas(modelo, pdf), index=pdf["_id_linha"].to_numpy()).reindex(ids).to_numpy()
    sc = pontuar_modelo_final(modelo, df, ["_id_linha"], parametros, funcao_score).toPandas()
    return pd.Series(sc["score"].to_numpy(), index=sc["_id_linha"].to_numpy()).reindex(ids).to_numpy()


def treinar_modelo_final(contexto, base, vencedora, parametros, funcao_automl=None, kwargs_automl=None):
    treino = spark.table(vencedora["tabela"])
    teste = base["teste"].drop("_id_linha")
    oot = base["oot_selecao"].drop("_id_linha")
    features, alvo, metrica = base["features"], base["alvo"], contexto["metrica"]
    if funcao_automl is None:
        log_pipeline("Modelo final: Optuna interno (LightGBM em pandas).")
        modelo = treinar_modelo_final_interno(treino, teste, features, alvo, metrica, parametros)
        return modelo, modelo["hiperparametros"], "optuna_interno"
    log_pipeline("Modelo final: AutoML do usuário.")
    modelo = funcao_automl(treino=treino, teste=teste, oot=oot, features=features, metrica=metrica,
                           n_trials=int(parametros["n_trials_optuna"]), **(kwargs_automl or {}))
    hiper = modelo.get("hiperparametros") if isinstance(modelo, dict) else None
    return modelo, hiper, "automl_usuario"

## Célula 5 — Estimativa final no OOT final

O OOT final é aberto **uma única vez**, depois de todas as escolhas. Com `forcar_reabertura_oot_final=False`, o bloco recusa uma segunda medição se a tabela `final_metricas` já existir, e recusa antes de treinar o modelo final, para não gastar o Optuna à toa; reabrir faz o OOT final virar uma base de seleção, e o número reportado deixa de ser uma estimativa honesta.

| Tabela | Conteúdo |
|---|---|
| `final_metricas` | KS, AUC-ROC, Gini, PR-AUC e a grade operacional no OOT final, com intervalo de 95% por bootstrap por chave |
| `final_por_safra` | As mesmas métricas por safra do OOT final |
| `final_operacional` | Para cada taxa de alerta: matriz de confusão no top k% e, ao lado, a taxa de alerta que o **corte de score fixado no teste** produz no OOT final (revela deriva do volume de alertas) |
| `final_matriz_ks` | Matriz de confusão no ponto do KS, com o corte escolhido no teste e aplicado ao OOT final |
| `final_calibracao` | Brier, *log loss* (perda logarítmica), ECE e média prevista × taxa observada, antes e depois da correção pelo β |
| `final_calibracao_faixas` | Média prevista e taxa observada em cada faixa de score corrigido |
| `final_psi` | PSI (*Population Stability Index*) do score entre teste e OOT final |
| `scores_oot_final` | Chave, safra, alvo, score e score corrigido, para o monitoramento |

**Correção pelo β** (Dal Pozzolo et al., 2015): p = β·s / (β·s − s + 1), em que s é o score do modelo treinado na base amostrada. Com β = 1 (baseline), p = s. A correção não muda a ordem dos scores: KS, AUC e PR-AUC ficam iguais; só as métricas de calibração mudam.

**ECE** (*Expected Calibration Error*, erro de calibração esperado): soma, nas faixas de score, de (peso da faixa) × |média prevista − taxa observada|.

**PSI**: Σ (p_oot − p_teste) × ln(p_oot / p_teste) nas faixas definidas pelos decis do score no teste. Referência usual: abaixo de 0,10 estável; de 0,10 a 0,25 atenção; acima de 0,25 mudança relevante.

In [ ]:
def corrigir_calibracao(scores, beta):
    s = np.clip(np.asarray(scores, dtype=np.float64), 1e-12, 1 - 1e-12)
    return beta * s / (beta * s - s + 1.0)


def medidas_calibracao(y, p, n_faixas):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    brier = float(np.mean((p - y) ** 2))
    log_loss = float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
    faixas = pd.qcut(pd.Series(p).rank(method="first"), n_faixas, labels=False)
    tabela = (pd.DataFrame({"faixa": faixas, "p": p, "y": y}).groupby("faixa")
              .agg(linhas=("y", "size"), media_prevista=("p", "mean"), taxa_observada=("y", "mean")).reset_index())
    ece = float(np.sum(tabela["linhas"] / len(y) * np.abs(tabela["media_prevista"] - tabela["taxa_observada"])))
    return {"brier": brier, "log_loss": log_loss, "ece": ece,
            "media_prevista": float(np.mean(p)), "taxa_observada": float(np.mean(y))}, tabela


def matriz_confusao(y, alerta):
    alerta = np.asarray(alerta, dtype=bool)
    y = np.asarray(y) == 1
    vp, fp = int(np.sum(alerta & y)), int(np.sum(alerta & ~y))
    fn, vn = int(np.sum(~alerta & y)), int(np.sum(~alerta & ~y))
    return {"alertas": vp + fp, "taxa_alerta": (vp + fp) / len(y), "VP": vp, "FP": fp, "FN": fn, "VN": vn,
            "revocacao": vp / builtins.max(vp + fn, 1), "precisao": vp / builtins.max(vp + fp, 1),
            "fpr": fp / builtins.max(fp + vn, 1)}


def alerta_top(scores, taxa):
    ordem = np.argsort(-np.asarray(scores), kind="mergesort")
    n = builtins.max(1, int(math.floor(taxa * len(scores))))
    alerta = np.zeros(len(scores), dtype=bool)
    alerta[ordem[:n]] = True
    return alerta, float(np.asarray(scores)[ordem[n - 1]])


def corte_ks(y, scores):
    ordem, fim = ordenar_scores(scores)
    y_ord = np.asarray(y, dtype=np.float64)[ordem]
    tp, fp = np.cumsum(y_ord)[fim], np.cumsum(1 - y_ord)[fim]
    i = int(np.argmax(tp / tp[-1] - fp / fp[-1]))
    return float(np.asarray(scores)[ordem][fim[i]])


def psi(scores_ref, scores_novo, n_faixas):
    bordas = np.unique(np.quantile(scores_ref, np.linspace(0, 1, n_faixas + 1)[1:-1]))
    bordas = np.r_[-np.inf, bordas, np.inf]
    ref = np.histogram(scores_ref, bordas)[0] / len(scores_ref)
    novo = np.histogram(scores_novo, bordas)[0] / len(scores_novo)
    ref, novo = np.clip(ref, 1e-6, None), np.clip(novo, 1e-6, None)
    faixas = pd.DataFrame({"faixa": range(len(ref)), "limite_inferior": bordas[:-1], "limite_superior": bordas[1:],
                           "proporcao_teste": ref, "proporcao_oot_final": novo,
                           "contribuicao": (novo - ref) * np.log(novo / ref)})
    return float(faixas["contribuicao"].sum()), faixas


def intervalos_bootstrap(y, codigos_chave, scores, taxas, n_bootstrap, semente):
    y = np.asarray(y, dtype=np.float64)
    ordem, fim = ordenar_scores(scores)
    y_ord = y[ordem]
    pontual = metricas_de_ordem(y_ord, np.ones_like(y_ord), fim, taxas)
    n_chaves = int(np.max(codigos_chave)) + 1
    rng = np.random.default_rng(semente)
    amostras = []
    for _ in range(n_bootstrap):
        pesos = rng.poisson(1.0, n_chaves).astype(np.float64)[codigos_chave][ordem]
        amostras.append(metricas_de_ordem(y_ord, pesos, fim, taxas))
    amostras = pd.DataFrame(amostras)
    return pd.DataFrame([{"metrica": m, "valor": pontual[m],
                          "ic_inferior": float(amostras[m].quantile(0.025)),
                          "ic_superior": float(amostras[m].quantile(0.975))} for m in pontual])


def verificar_trava_oot_final(contexto, base, parametros):
    """Recusa medir o OOT final de novo, salvo com forcar_reabertura_oot_final=True."""
    registro = contexto["registro"]
    tabela_guarda = nome_tabela(base["destino"], base["prefixo"], "final_metricas")
    if not parametros["forcar_reabertura_oot_final"]:
        try:
            spark.table(tabela_guarda).schema
            raise RuntimeError(
                f"O OOT final já foi medido ({tabela_guarda} existe). Reabrir transforma o OOT final em base de "
                "seleção. Se for mesmo necessário, use forcar_reabertura_oot_final=True e registre isso na documentação.")
        except RuntimeError:
            raise
        except Exception:
            pass
    else:
        try:
            spark.table(tabela_guarda).schema
            registro["avisos"].append("O OOT final foi reaberto com forcar_reabertura_oot_final=True.")
        except Exception:
            pass


def estimar_oot_final(contexto, base, modelo, vencedora, parametros, funcao_score=None):
    destino, prefixo, registro = base["destino"], base["prefixo"], contexto["registro"]
    taxas = parametros["taxas_alerta_relatorio"]
    info_final = _info_avaliacao(base["oot_final"], contexto["colunas"])
    info_teste = base["info_teste"]
    log_pipeline("Pontuando o teste e o OOT final com o modelo final.")
    s_teste = scores_alinhados(modelo, base["teste"], info_teste["ids"], parametros, funcao_score)
    s_final = scores_alinhados(modelo, base["oot_final"], info_final["ids"], parametros, funcao_score)
    y, y_teste = info_final["y"], info_teste["y"]

    # Métricas com intervalo
    metricas = intervalos_bootstrap(y, info_final["codigos_chave"], s_final, taxas,
                                    parametros["n_bootstrap"], parametros["semente"])
    metricas.insert(0, "base", "oot_final")
    m_teste = calcular_metricas(y_teste, s_teste, None, taxas)
    metricas["valor_teste"] = metricas["metrica"].map(m_teste)

    # Por safra
    linhas = []
    for safra in pd.unique(info_final["safra"]):
        mascara = info_final["safra"] == safra
        m = calcular_metricas(y[mascara], s_final[mascara], None, taxas)
        linhas.append({"safra": safra, "linhas": int(mascara.sum()), "positivos": int(y[mascara].sum()),
                       "prevalencia": float(y[mascara].mean()), **m})
    por_safra = pd.DataFrame(linhas).sort_values("safra").reset_index(drop=True)

    # Operacional
    operacional = []
    for taxa in taxas:
        alerta, _ = alerta_top(s_final, taxa)
        _, limiar_teste = alerta_top(s_teste, taxa)
        linha = {"taxa_alerta_alvo": taxa, **matriz_confusao(y, alerta)}
        linha["lift"] = linha["precisao"] / builtins.max(float(np.mean(y)), 1e-12)
        linha["corte_score_teste"] = limiar_teste
        linha["taxa_alerta_com_corte_do_teste"] = float(np.mean(s_final >= limiar_teste))
        operacional.append(linha)
    operacional = pd.DataFrame(operacional)

    # Ponto do KS
    corte = corte_ks(y_teste, s_teste)
    matriz_ks = pd.DataFrame([{"base": "teste", "corte_score": corte, **matriz_confusao(y_teste, s_teste >= corte)},
                              {"base": "oot_final", "corte_score": corte, **matriz_confusao(y, s_final >= corte)}])

    # Calibração
    beta = float(vencedora.get("beta", 1.0))
    p_final = corrigir_calibracao(s_final, beta)
    antes, _ = medidas_calibracao(y, np.asarray(s_final, dtype=np.float64), parametros["n_faixas_calibracao"])
    depois, faixas_cal = medidas_calibracao(y, p_final, parametros["n_faixas_calibracao"])
    calibracao = pd.DataFrame([{"score": "original", "beta": 1.0, **antes},
                               {"score": "corrigido", "beta": beta, **depois}])

    # PSI
    valor_psi, faixas_psi = psi(s_teste, s_final, parametros["n_faixas_psi"])
    faixas_psi.insert(0, "psi_total", valor_psi)

    # Scores para monitoramento
    col = contexto["colunas"]
    scores_final = pd.DataFrame({"_id_linha": info_final["ids"], col["chave"]: info_final["chave"],
                                 col["safra"]: info_final["safra_original"], col["alvo"]: y.astype(int),
                                 "score": s_final, "score_corrigido": p_final})

    tabelas = {}
    for nome, pdf in (("final_por_safra", por_safra), ("final_operacional", operacional),
                      ("final_matriz_ks", matriz_ks), ("final_calibracao", calibracao),
                      ("final_calibracao_faixas", faixas_cal), ("final_psi", faixas_psi),
                      ("scores_oot_final", scores_final), ("final_metricas", metricas)):
        salvar_tabela(spark.createDataFrame(pdf), destino, prefixo, nome, registro)
        tabelas[nome] = nome_tabela(destino, prefixo, nome)

    nome_m = NOME_METRICA[contexto["metrica"]]
    linha_m = metricas[metricas["metrica"] == nome_m].iloc[0]
    log_pipeline(f"OOT final: {contexto['metrica']} = {linha_m['valor']:.4f} "
                 f"[{linha_m['ic_inferior']:.4f}; {linha_m['ic_superior']:.4f}] | PSI teste × OOT final = {valor_psi:.3f}")
    return {"metricas": metricas, "por_safra": por_safra, "operacional": operacional, "matriz_ks": matriz_ks,
            "calibracao": calibracao, "calibracao_faixas": faixas_cal, "psi": valor_psi, "psi_faixas": faixas_psi,
            "tabelas": tabelas}

## Célula 6 — Orquestração e registro

| Função | O que faz |
|---|---|
| `decidir_tecnica(contexto)` | Só a decisão (célula 3). Útil para revisar a escolha antes de treinar o modelo final |
| `treinar_e_estimar(contexto, funcao_automl=None, funcao_score=None, kwargs_automl=None)` | Modelo final e estimativa no OOT final (células 4 e 5). Exige a decisão já feita |
| `executar_bloco3(...)` | As duas, em sequência |

No fim, o registro completo da execução (parâmetros de todos os blocos, versões, avisos, tabelas, decisão, técnica vencedora, hiperparâmetros e métricas finais) é gravado como JSON na tabela `registro_execucao`.

Chaves acrescentadas ao `contexto`: `parametros_bloco3`, `decisao_resultados`, `decisao_resumo`, `configuracoes_decisao`, `tecnica_vencedora`, `modelo_final`, `hiperparametros_finais`, `origem_modelo_final`, `resultado_final`.

In [ ]:
def decidir_tecnica(contexto, **sobrescritos):
    if not contexto.get("modulo_ativo"):
        log_pipeline("Módulo de desbalanceamento inativo: o bloco 3 não foi executado.")
        return contexto
    parametros = montar_parametros_bloco3(contexto, **sobrescritos)
    base = preparar_bases_bloco3(contexto, parametros)
    configuracoes, tabelas = executar_grade_decisao(contexto, base, parametros)
    resultados, resumo, vencedora = resumir_decisao(contexto, base, parametros, tabelas)
    gravavel = resumo.astype({"nivel": "float64"}).assign(motivos_reprovacao=resumo["motivos_reprovacao"].fillna(""))
    salvar_tabela(spark.createDataFrame(gravavel), base["destino"], base["prefixo"],
                  "decisao_resumo", contexto["registro"])
    contexto.update({"parametros_bloco3": parametros, "_base_bloco3": base, "decisao_resultados": resultados,
                     "decisao_resumo": resumo, "configuracoes_decisao": configuracoes, "tecnica_vencedora": vencedora})
    contexto["registro"]["decisao"] = {"configuracoes": configuracoes, "tabelas": tabelas, "vencedora": vencedora}
    return contexto


def _gravar_registro(contexto):
    base = contexto["_base_bloco3"]
    registro = dict(contexto["registro"])
    registro["fim"] = datetime.now().isoformat(timespec="seconds")
    texto = json.dumps(registro, default=str, ensure_ascii=False)
    salvar_tabela(spark.createDataFrame(pd.DataFrame({"registro_json": [texto], "momento": [registro["fim"]]})),
                  base["destino"], base["prefixo"], "registro_execucao", contexto["registro"])


def treinar_e_estimar(contexto, funcao_automl=None, funcao_score=None, kwargs_automl=None):
    if "tecnica_vencedora" not in contexto:
        raise ValueError("Execute decidir_tecnica(contexto) antes.")
    parametros, base, vencedora = contexto["parametros_bloco3"], contexto["_base_bloco3"], contexto["tecnica_vencedora"]
    verificar_trava_oot_final(contexto, base, parametros)
    modelo, hiper, origem = treinar_modelo_final(contexto, base, vencedora, parametros, funcao_automl, kwargs_automl)
    resultado = estimar_oot_final(contexto, base, modelo, vencedora, parametros, funcao_score)
    contexto.update({"modelo_final": modelo, "hiperparametros_finais": hiper, "origem_modelo_final": origem,
                     "resultado_final": resultado})
    contexto["registro"]["modelo_final"] = {
        "origem": origem, "tecnica": vencedora["tecnica"], "nivel": vencedora["nivel"], "beta": vencedora.get("beta"),
        "tabela_treino": vencedora["tabela"], "hiperparametros": hiper,
        "metricas_oot_final": resultado["metricas"].to_dict(orient="records"), "psi": resultado["psi"],
    }
    _gravar_registro(contexto)
    return contexto


def executar_bloco3(contexto, funcao_automl=None, funcao_score=None, kwargs_automl=None, **sobrescritos):
    contexto = decidir_tecnica(contexto, **sobrescritos)
    if not contexto.get("modulo_ativo"):
        return contexto
    return treinar_e_estimar(contexto, funcao_automl, funcao_score, kwargs_automl)

## Célula 7 — Exemplo de uso

Recomendo rodar em duas etapas: primeiro a decisão, conferir o resumo, e só depois o modelo final e o OOT final.

In [ ]:
# # 1. Decisão no OOT de seleção
# contexto = decidir_tecnica(contexto)
# display(contexto["decisao_resumo"])
# print(contexto["tecnica_vencedora"])
#
# # 2a. Modelo final com o Optuna interno
# contexto = treinar_e_estimar(contexto)
#
# # 2b. Ou com o seu AutoML (ajuste os nomes dos argumentos com uma função pequena, se necessário)
# # def meu_automl(treino, teste, oot, features, metrica, n_trials, **kwargs):
# #     return automl_interno(df_treino=treino, df_teste=teste, df_oot=oot, features=features,
# #                           metrica=metrica, n_trials=n_trials, **kwargs)
# # def meu_score(modelo, df):
# #     return ...  # o mesmo df com a coluna "score" (probabilidade da classe positiva)
# # contexto = treinar_e_estimar(contexto, funcao_automl=meu_automl, funcao_score=meu_score)
#
# r = contexto["resultado_final"]
# display(r["metricas"])       # OOT final com intervalo de 95%
# display(r["por_safra"])
# display(r["operacional"])
# display(r["matriz_ks"])
# display(r["calibracao"])
# print("PSI teste × OOT final:", r["psi"])